In [8]:
import warnings

import numpy as np
from ipynb.fs.defs.ARMA import arma_residuals, fit_arma
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# ARMA-GARCH Model
- Define and Minimise Negative Log-Likelihood of ARMA-GARCH
- Forecast Next Day Mean and Volatility using ARMA-GARCH
- Find Returns Confidence Intervals using Forecasts


In [10]:
def negloglik_armagarch(params, ret, p, q, forecast=False):
    '''Negative Log-Likelihood of ARMA-GARCH Model'''
    
    # Mean Estimation
    res = fit_arma(ret, p, q)
    resid, next = arma_residuals(res, ret, p, q, True)
    fitted = ret - resid
    
    alpha_0, alpha_1, beta_1 = params
    
    # Compute Errors
    errors = ret - fitted
    
    # Initialise Variance
    vars = np.zeros(len(ret))
    vars[0] = np.var(errors)
    
    # Compute Variances
    for i in range(1, len(ret)):
        vars[i] = alpha_0 + alpha_1 * errors[i-1]**2 + beta_1 * vars[i-1]
        
    loglik = -0.5 * np.sum(np.log(2 * np.pi) + np.log(vars) + errors**2 / vars)
    
    # Forecast Next Mean and Sigma
    mu = next
    
    # Forecast Next Volatility
    sigma = alpha_0 + alpha_1 * errors[-1]**2 + beta_1 * vars[-1]
    
    if forecast:
        return -loglik, mu, np.sqrt(sigma)
    else:
        return -loglik

In [9]:
def confint(mu, sigma, confidence=0.95):
    '''Create Confidence Intervals from Mean and Volatility'''
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha/2)
    margin = z * (sigma)
    
    lower = mu - margin
    upper = mu + margin
    
    return lower, upper

In [ ]:
def armagarch_forecast(ret, init, bounds, p, q):
    '''Forecast Next Day Mean and Volatility using ARMA-GARCH'''
    res = minimize(
        negloglik_armagarch,
        x0=init,
        args=(ret*1000, p, q),
        bounds=bounds,
        tol=10**-10
    )

    [alpha_0, alpha_1, beta_1] = res.x
    alpha_0 /= 1000000

    loglik, mu, sigma = negloglik_armagarch([alpha_0, alpha_1, beta_1], ret, p, q, True)
    lower, upper = confint(mu, sigma)
    return alpha_1, beta_1, mu, sigma, lower, upper